# World Models | Embodied / Physical

In [1]:
# World Model: Simple Grid World Simulator
from dataclasses import dataclass
from typing import Tuple, List, Dict

In [2]:
@dataclass
class WorldState:
    agent_pos: Tuple[int, int]
    goal_pos: Tuple[int, int]
    obstacles: set
    grid_size: int = 10

class WorldModel:
    """Internal world model for planning without real execution."""

    def __init__(self, initial_state: WorldState):
        self.state = initial_state
        self.actions = {"up": (0, 1), "down": (0, -1), "left": (-1, 0), "right": (1, 0)}

    def predict(self, state: WorldState, action: str) -> WorldState:
        """Predict the next state given an action (imagination)."""
        dx, dy = self.actions.get(action, (0, 0))
        new_x = max(0, min(state.grid_size - 1, state.agent_pos[0] + dx))
        new_y = max(0, min(state.grid_size - 1, state.agent_pos[1] + dy))
        new_pos = (new_x, new_y)
        if new_pos in state.obstacles:
            new_pos = state.agent_pos  # Blocked
        return WorldState(
            agent_pos=new_pos, goal_pos=state.goal_pos,
            obstacles=state.obstacles, grid_size=state.grid_size,
        )

    def reward(self, state: WorldState) -> float:
        if state.agent_pos == state.goal_pos:
            return 10.0
        # Manhattan distance penalty
        dist = abs(state.agent_pos[0] - state.goal_pos[0]) + abs(state.agent_pos[1] - state.goal_pos[1])
        return -dist * 0.1

    def plan(self, state: WorldState, depth: int = 5) -> List[str]:
        """Plan by simulating actions in imagination."""
        best_plan = []
        best_reward = float('-inf')

        def search(current: WorldState, plan: List[str], d: int):
            nonlocal best_plan, best_reward
            if d == 0 or current.agent_pos == current.goal_pos:
                r = self.reward(current)
                if r > best_reward:
                    best_reward = r
                    best_plan = list(plan)
                return
            for action in self.actions:
                next_state = self.predict(current, action)
                search(next_state, plan + [action], d - 1)

        search(state, [], depth)
        return best_plan

In [3]:
# Usage
initial = WorldState(
    agent_pos=(0, 0), goal_pos=(3, 3),
    obstacles={(1, 1), (2, 1), (1, 2)},
)
wm = WorldModel(initial)
plan = wm.plan(initial, depth=6)
print(f"Plan: {plan}")

# Simulate the plan
state = initial
for action in plan:
    state = wm.predict(state, action)
    print(f"  {action} -> {state.agent_pos}")
print(f"Reached goal: {state.agent_pos == state.goal_pos}")

# Compare planned vs random baseline (actual simulation)
import random
random_steps = 0
pos = initial.agent_pos
random.seed(42)
for _ in range(50):  # cap at 50 random steps
    action = random.choice(list(wm.actions.keys()))
    next_state = wm.predict(WorldState(pos, initial.goal_pos, initial.obstacles), action)
    pos = next_state.agent_pos
    random_steps += 1
    if pos == initial.goal_pos:
        break
random_result = "reached goal" if pos == initial.goal_pos else "did NOT reach goal"
print(f"\nPlanned: {len(plan)} steps (reached goal) | Random baseline: {random_steps} steps ({random_result})")

Plan: ['up', 'up', 'up', 'right', 'right', 'right']
  up -> (0, 1)
  up -> (0, 2)
  up -> (0, 3)
  right -> (1, 3)
  right -> (2, 3)
  right -> (3, 3)
Reached goal: True

Planned: 6 steps (reached goal) | Random baseline: 50 steps (did NOT reach goal)
